# 06 — AKC/OODA Phase Observability

## Research Question
Can telemetry infer cognitive threat phases in the Agentic Kill Chain / OODA framing?

## Hypothesis
Telemetry can operationalize cognitive threat phases, making multi-agent failures interpretable rather than only detectable.

## Input Data
- `episode_df_all`
- `early_df_all` for temporal AKC experiments

## Prediction/Analysis Target
- `akc_phase` and related OODA-style phase labels

## Validation Protocol
Leave-one-seed-out, leave-one-attack-type-out, and prefix-level phase classification.

## Expected Paper Artifact
- AKC phase confusion matrices
- Temporal AKC curves
- Leave-one-attack-type-out phase generalization table


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


The Agentic Kill Chain: A Cognitive Threat Framework for Autonomous Agent Ecosystems

The Agentic Kill Chain (AKC) Framework mapped to the OODA Loop. Each attack phase targets a corresponding stage of the agent’s cognitive cycle: Semantic Infection compromises perception (Observe), Cognitive Compromise corrupts reasoning (Orient/Decide), Agency Propagation exploits coordination (Act), and
Systemic Execution materializes impact in the physical/digital world.

When multiple agents are compromised, Kavathekar et al. [26] show that colluding agents can bypass consensus safeguards: when agents use consensus-based protections (e.g., requiring agreement from multiple agents before executing high-risk actions), a “conspiracy” of compromised agents can provide mutually reinforcing false confirmations that bypass these protections, effectively enabling attacks that would be blocked by single-agent safety checks.

Kavathekar et al. [26] demonstrate that Byzantine behaviour patterns can destabilize collaborative processes without overt malicious action: compromised agents inject noise or contradictory information to derail consensus mechanisms in multi-agent settings, effectively launching a “Cognitive Denial of Service” that prevents the agent collective from accomplishing legitimate objectives.

Trace analysis reveals exploitation of the A2A protocol’s identity inheritance mechanism (Section 3.3): the Personal Agent fragments the malicious
transaction request into three semantically benign messages —«verify wallet balance», «check transaction limits», «process optimization transfer»— and routes them through a Logistics Agent [26]. This implements the propagation objective 𝜎(𝒢,𝑣0,𝑡) by maximizing infected node count while minimizing
detection probability. 

Telemetry can operationalize cognitive threat phases in LLM-based multi-agent systems.

--------------------------------------------------------------------------------------

Semantic Infection

Observe

Prompt/observation contamination

injection markers, semantic drift, suspicious instruction propagation

--------------------------------------------------------------------------------------

Cognitive Compromise

Orient/Decide

raciocínio corrompido ou contraditório

contradiction markers, semantic variance, disagreement, answer changes

--------------------------------------------------------------------------------------

Agency Propagation

Act

propagação entre agentes

tool-call propagation, compromised agent influence, similarity entre agentes

--------------------------------------------------------------------------------------

Systemic Execution

Impact

execução concreta da ação

suspicious tool call, attack_tool_invoked, unsafe output

### AKC balanceado/removendo systemic_execution

Objetivo: avaliar se a classificação de fase melhora quando removemos a classe extremamente subrepresentada systemic_execution.

In [ ]:
OUTPUT_DIR = Path("results/tamas/paper_ready")

In [ ]:
AKC_METADATA_COLS = {
    "is_attack",
    "attack_type",
    "akc_phase",
    "seed",
    "task_id",
    "episode_id",
    "scenario",
    "architecture",
    "model_name",
    "benchmark",
    "expected_label",
    "source_file",
    "label",
    "target",
}

episode_df_all = add_akc_phase_from_attack_type(
    episode_df_all,
    attack_col="attack_type",
    output_col="akc_phase",
)

print(episode_df_all["akc_phase"].value_counts(dropna=False))

print(
    episode_df_all
    .groupby(["attack_type", "akc_phase"])
    .size()
)

episode_df_all = normalize_seed_column(episode_df_all)

AKC_FEATURES = infer_telemetry_features(episode_df_all, AKC_METADATA_COLS)

akc_all_df = prepare_akc_phase_df(
    episode_df_all,
    phase_col="akc_phase",
    drop_phases=["systemic_execution"],
)

akc_all_results = run_akc_phase_leave_one_seed(
    df=akc_all_df,
    features=AKC_FEATURES,
    phase_col="akc_phase",
    model_kind="rf",
    experiment_name="akc_all_phases",
)

akc_all_summary = summarize_multiclass_results(
    akc_all_results,
    group_cols=["experiment", "model"],
)

akc_all_results.to_csv(OUTPUT_DIR / "akc_all_phases_leave_one_seed_raw.csv", index=False)
akc_all_summary.to_csv(OUTPUT_DIR / "akc_all_phases_leave_one_seed_summary.csv", index=False)

akc_all_summary

### Matriz de confusão por ACK fase

Objetivo: entender quais fases são confundidas com quais.

In [ ]:
akc_pred_df = collect_akc_phase_predictions_leave_one_seed(
    df=akc_all_df,
    features=AKC_FEATURES,
    phase_col="akc_phase",
    model_kind="rf",
    experiment_name="akc_without_systemic_confusion",
)

cm_norm = plot_confusion_matrix_from_predictions(
    akc_pred_df,
    normalize=True,
    output_path=OUTPUT_DIR / "akc_phase_confusion_matrix_normalized.png",
    title="AKC Phase Confusion Matrix - Normalized",
)

cm_norm

In [ ]:
print(
    classification_report(
        akc_pred_df["true_phase"],
        akc_pred_df["pred_phase"],
        zero_division=0,
    )
)

### Leave-one-attack-type-out para fase

Objetivo: testar se o classificador aprende fase cognitiva ou apenas memoriza attack_type.

Exemplo: treinar em DPI e IPI como semantic_infection, testar em impersonation como ataque não visto da mesma fase.

In [ ]:
akc_loao_results, akc_loao_pred_df = run_akc_phase_leave_one_attack_type_out(
    df=episode_df_all,
    features=AKC_FEATURES,
    phase_col="akc_phase",
    attack_col="attack_type",
    benign_label="benign",
    model_kind="rf",
    drop_phases=["systemic_execution"],
)

akc_loao_results

In [ ]:
# akc_loao_phase_summary = (
#     akc_loao_results
#     .groupby("true_phase_of_heldout_attack")
#     [
#         [
#             "accuracy",
#             "balanced_accuracy",
#             "f1_macro",
#             "f1_weighted",
#         ]
#     ]
#     .agg(["mean", "std"])
#     .reset_index()
# )

# akc_loao_phase_summary.columns = [
#     "_".join(col).strip("_") if isinstance(col, tuple) else col
#     for col in akc_loao_phase_summary.columns
# ]

# akc_loao_phase_summary

In [ ]:
cm_loao_norm = plot_confusion_matrix_from_predictions(
    akc_loao_pred_df,
    true_col="true_phase",
    pred_col="pred_phase",
    normalize=True,
    output_path=OUTPUT_DIR / "akc_loao_confusion_matrix_normalized.png",
    title="AKC Phase Leave-One-Attack-Type-Out - Normalized",
)

cm_loao_norm

### Fases temporais/prefix-level usando early_df_all

Objetivo: avaliar se fases da Kill Chain ficam detectáveis antes do fim do workflow.

In [ ]:
EARLY_AKC_METADATA_COLS = AKC_METADATA_COLS | {
    "prefix_steps",
    "total_steps",
    "observed_step_ratio",
}

In [ ]:
early_df_all = normalize_seed_column(early_df_all)
early_df_all["is_attack"] = early_df_all["is_attack"].astype(int)

EARLY_AKC_FEATURES_ALL = infer_early_akc_features(
    early_df_all,
    remove_prefix_progress=False,
)

EARLY_AKC_FEATURES_NO_PROGRESS = infer_early_akc_features(
    early_df_all,
    remove_prefix_progress=True,
)

In [ ]:
early_df_all = add_akc_phase_from_attack_type(
    early_df_all,
    attack_col="attack_type",
    output_col="akc_phase",
)

temporal_akc_all_results, temporal_akc_all_pred = run_temporal_akc_phase_classification(
    early_df=early_df_all,
    features=EARLY_AKC_FEATURES_ALL,
    phase_col="akc_phase",
    model_kind="rf",
    drop_phases=["systemic_execution"],
    experiment_name="temporal_akc_all_prefix_features",
)

temporal_akc_all_summary = summarize_multiclass_results(
    temporal_akc_all_results,
    group_cols=["experiment", "trace_fraction"],
)

plot_temporal_akc_curves(
    temporal_akc_all_summary,
    metric="balanced_accuracy_mean",
    output_path=OUTPUT_DIR / "temporal_akc_balanced_accuracy.png",
)

In [ ]:
temporal_75_pred = temporal_akc_all_pred[
    temporal_akc_all_pred["trace_fraction"] == 0.75
].copy()

cm_temporal_75 = plot_confusion_matrix_from_predictions(
    temporal_75_pred,
    true_col="true_phase",
    pred_col="pred_phase",
    normalize=True,
    output_path=OUTPUT_DIR / "temporal_akc_075_confusion_matrix_normalized.png",
    title="Temporal AKC Phase Classification at 75% - Normalized",
)

cm_temporal_75

### Experimento AKC-1 — Classificar fase da Kill Chain via telemetria

In [ ]:
# episode_akc_1 = add_akc_ooda_labels(episode_df_all)

# display(
#     episode_akc_1.groupby(["attack_type", "akc_phase", "ooda_stage"])
#     .size()
#     .reset_index(name="n")
# )

### Experimento AKC-2 — Telemetria prediz fase cognitiva?

In [ ]:
# akc_phase_result = evaluate_multiclass_target(
#     episode_akc_1,
#     target_col="akc_phase",
#     numeric_features=TELEMETRY_NUMERIC,
#     model_kind="rf",
# )

# akc_phase_result

### Experimento AKC-3 — Assinaturas de telemetria por fase

In [ ]:
# akc_summary = summarize_akc_telemetry(episode_akc_1)
# akc_summary

Interpretação esperada:

* semantic_infection: mais injection markers;

* cognitive_compromise: mais contradiction markers e variância semântica;

* agency_propagation: maior similaridade entre agentes ou propagação de tool calls;

* systemic_execution: maior taxa de tools suspeitas.

### Experimento AKC-4 — Collusion bypass e Byzantine cognitive DoS

Collusion bypass

Hipótese:

Colluding agents increase semantic agreement while producing unsafe convergence.

In [ ]:
# episode_df_all["collusive_convergence_score"] = (
#     episode_df_all["avg_pairwise_message_similarity"].fillna(0)
#     * episode_df_all["num_tool_calls"].fillna(0)
# )

# display(
#     episode_df_all.groupby("attack_type")
#     .agg(
#         avg_similarity=("avg_pairwise_message_similarity", "mean"),
#         avg_tool_calls=("num_tool_calls", "mean"),
#         collusive_score=("collusive_convergence_score", "mean"),
#     )
#     .reset_index()
#     .sort_values("collusive_score", ascending=False)
# )

Byzantine Cognitive Denial of Service

Hipótese:

Byzantine agents destabilize workflows by increasing contradiction, latency, and token cost without necessarily invoking explicit malicious tools.

In [ ]:
# episode_df_all["cognitive_dos_score"] = (
#     episode_df_all["contradiction_marker_count"].fillna(0)
#     + episode_df_all["var_pairwise_message_similarity"].fillna(0)
#     + np.log1p(episode_df_all["latency_total_s"].fillna(0))
#     + np.log1p(episode_df_all["total_tokens"].fillna(0))
# )

# display(
#     episode_df_all.groupby("attack_type")
#     .agg(
#         cognitive_dos_score=("cognitive_dos_score", "mean"),
#         contradictions=("contradiction_marker_count", "mean"),
#         semantic_var=("var_pairwise_message_similarity", "mean"),
#         latency=("latency_total_s", "mean"),
#         tokens=("total_tokens", "mean"),
#     )
#     .reset_index()
#     .sort_values("cognitive_dos_score", ascending=False)
# )